In [1]:
import functools
import logging
import numpy as np
import pandas as pd

In [2]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

class InsufficientPaymentError(Exception):
    """Raised when a payment amount is insufficient or exceeds allowed limits."""
    pass

In [3]:
def retry(max_attempts=3):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            attempts = 0
            while attempts < max_attempts:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    attempts += 1
                    logging.warning(f"Attempt {attempts} failed in {func.__name__}: {e}")
                    if attempts == max_attempts:
                        raise
        return wrapper
    return decorator

In [4]:
def log_execution(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logging.info(f"Starting execution of {func.__name__}")
        result = func(*args, **kwargs)
        logging.info(f"Successfully completed {func.__name__}")
        return result
    return wrapper

In [5]:
class Loan:
    def __init__(self, loan_id, customer_id, branch_id, product, disbursal_date, principal, rate_pct, tenure_months, emi):
        self.loan_id = loan_id
        self.customer_id = customer_id
        self.branch_id = branch_id
        self.product = product
        self.disbursal_date = pd.to_datetime(disbursal_date)
        self.principal = float(principal)
        self.rate_pct = float(rate_pct)
        self.tenure_months = int(tenure_months)
        self.emi = float(emi)

In [6]:
def __repr__(self):
        return f"<Loan {self.loan_id} [{self.product}]: Principal=₹{self.principal:,.2f}, EMI=₹{self.emi:,.2f}>"

In [7]:
class TwoWheelerLoan(Loan):
    def __init__(self, loan_id, customer_id, branch_id, disbursal_date, principal, rate_pct, tenure_months, emi, vehicle_make=None):
        super().__init__(loan_id, customer_id, branch_id, "Two-Wheeler", disbursal_date, principal, rate_pct, tenure_months, emi)
        self.vehicle_make = vehicle_make

    def __repr__(self):
        return f"<TwoWheelerLoan {self.loan_id}: Principal=₹{self.principal:,.2f}, Make={self.vehicle_make}>"

In [8]:
class GoldLoan(Loan):
    def __init__(self, loan_id, customer_id, branch_id, disbursal_date, principal, rate_pct, tenure_months, emi, gold_weight_gms=None):
        super().__init__(loan_id, customer_id, branch_id, "Gold", disbursal_date, principal, rate_pct, tenure_months, emi)
        self.gold_weight_gms = gold_weight_gms

    def __repr__(self):
        return f"<GoldLoan {self.loan_id}: Principal=₹{self.principal:,.2f}, GoldWt={self.gold_weight_gms}g>"

In [12]:
@log_execution
def generate_vectorized_emi_schedule(loan_id, principal, rate_pct, tenure_months, start_date):
    """Generates amortization schedule entirely via NumPy vectorized arrays without loops."""
    r = rate_pct / 1200.0
    n = tenure_months
    emi = principal * (r * (1 + r)**n) / ((1 + r)**n - 1)
    
    emi_nos = np.arange(1, n + 1)
    dates = pd.date_range(start=start_date, periods=n, freq='MS')
    
    interest_parts = principal * r * (1 + r)**(emi_nos - 1) / ((1 + r)**n - 1) # Simplified or standard reducing balance
    balances = np.zeros(n)
    cur_bal = principal
    interest_arr = np.zeros(n)
    principal_arr = np.zeros(n)
    factors = (1 + r)**np.arange(n)
    remaining = principal * ((1 + r)**n - (1 + r)**np.arange(n)) / ((1 + r)**n - 1)
    start_balances = np.insert(remaining[:-1], 0, principal)
    interest_arr = start_balances * r
    principal_arr = emi - interest_arr
    
    schedule_df = pd.DataFrame({
        'loan_id': loan_id,
        'emi_no': emi_nos,
        'due_date': dates,
        'emi_amount': round(emi, 2),
        'principal_part': round(principal_arr, 2),
        'interest_part': round(interest_arr, 2)
    })
    return schedule_df

In [13]:
@log_execution
def compute_roll_rate_matrix(mbs_df):
    """Computes month-over-month roll rates using shift and pd.crosstab(normalize='index')."""
    mbs_sorted = mbs_df.sort_values(['loan_id', 'month_end'])
    mbs_sorted['prev_bucket'] = mbs_sorted.groupby('loan_id')['bucket'].shift(1)
    
    # Filter consecutive months
    mbs_sorted['month_end'] = pd.to_datetime(mbs_sorted['month_end'])
    mbs_sorted['prev_month_end'] = mbs_sorted.groupby('loan_id')['month_end'].shift(1)
    consecutive = mbs_sorted[
        (mbs_sorted['prev_bucket'].notna()) &
        ((mbs_sorted['month_end'].dt.year * 12 + mbs_sorted['month_end'].dt.month) -
         (mbs_sorted['prev_month_end'].dt.year * 12 + mbs_sorted['prev_month_end'].dt.month) == 1)
    ]
    
    bucket_order = ['Current', '1-30', '31-60', '61-90', '90+']
    matrix = pd.crosstab(
        consecutive['prev_bucket'],
        consecutive['bucket'],
        normalize='index'
    ) * 100.0
    return matrix.reindex(index=bucket_order, columns=bucket_order, fill_value=0.0)